## 10b 外卖场景：公服设施上下文（格网）

**笔记本**：`10b_waimai_facilities_context.ipynb`

**库**：`geopandas`、`pandas`、`numpy`、`scipy.spatial.cKDTree`、`shapely` 等。

**输入**：
- `../08 POI Demand/data_out/sz_demand_grid.gpkg`（与主分析同 `h3_id`，H3 res=8）
- `../00/11-public-facilities/SHP/*.shp`（充电/换电/上下客区/步行街等）
- `../00/04-transport-download-data/2-transport-facilities/...`（公交、地铁，与 10 一致）
- `../05 Barrier Layers/data_out/sz_barrier_bridges.gpkg`、`sz_barrier_tunnels.gpkg`（与公服桥隧点 **空间对照**，不替代 05 摩擦）

**做了什么 / 算了什么**：
- **补能**：仅电动车相关（充电/换电/电动自行车充电）；`其它能源站` 仅保留名称/属性含「充电/换电/新能源」的记录。
- **停车**：仅 **上下客区** 作为路缘装卸弱代理。
- **公交/地铁**：以 00 交通设施 SHP 为主；公服同名类可做校验，不在格网重复一套距离指标。
- **通行形态**：步行街/特殊通道/通行设施等 **格内计数**；投影坐标下 **80 m buffer 并集**，判断格网质心是否落入步行相关缓冲带。
- **桥隧立交**：与 05 分工——摩擦仍以 `sz_barrier_*` 为准；此处只做与公服桥隧/立交点的 **最近距离、缓冲命中** 等上下文字段。

**写出文件**：
- `data_out/sz_waimai_context_grid.gpkg`

**典型输出信息**：格网单元数、各设施图层加载/跳过提示、各 `*_cnt` 列非零格数、`to_file` 保存路径。


In [ ]:
# ============================================================
#  路径与网格（与 08 demand 网格一致）
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from shapely.ops import unary_union
import warnings

warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

DEMAND_GRID = Path("../08 POI Demand/data_out/sz_demand_grid.gpkg")
GFS = Path("../00/11-public-facilities/SHP")

# 主数据：与 10 一致
BUS_STOPS_PRIMARY = Path("../00/04-transport-download-data/2-transport-facilities/bus/shenzhen_bus_stops.shp")
METRO_STOPS_PRIMARY = Path("../00/04-transport-download-data/2-transport-facilities/metro/opendata/shenzhen_metro_stations_opendata.shp")

BARRIER_BRIDGES = Path("../05 Barrier Layers/data_out/sz_barrier_bridges.gpkg")
BARRIER_TUNNELS = Path("../05 Barrier Layers/data_out/sz_barrier_tunnels.gpkg")

PROJ_METRIC = "EPSG:32650"  # WGS 84 / UTM 50N，深圳附近米制缓冲

if not DEMAND_GRID.exists():
    raise FileNotFoundError(f"请先运行 08 生成 {DEMAND_GRID}")

grid = gpd.read_file(DEMAND_GRID)[["h3_id", "geometry"]].to_crs(4326)
print(f"Grid cells: {len(grid):,}")


def count_in_grid(gdf: gpd.GeoDataFrame, col: str) -> pd.Series:
    """点落入格网 polygon 内计数。"""
    if gdf is None or len(gdf) == 0:
        return pd.Series(0, index=grid["h3_id"])
    gdf = gdf.to_crs(4326)
    j = gpd.sjoin(gdf, grid, how="inner", predicate="within")
    c = j.groupby("h3_id").size()
    return c.reindex(grid["h3_id"], fill_value=0)


def load_points(stem: str, quiet: bool = False):
    """stem: shapefile basename without .shp (ASCII), e.g. shenzhen_chong_dian_zhan"""
    p = GFS / f"{stem}.shp"
    if not p.exists():
        if not quiet:
            print(f"  (skip) missing {p.name}")
        return None
    return gpd.read_file(p)


In [ ]:
# ============================================================
#  1) 电动车补能 + 上下客区 → 格网计数
# ============================================================

ev_layers = {
    "ev_charge_cnt": "shenzhen_chong_dian_zhan",
    "ev_dedicated_charge_cnt": "shenzhen_zhuan_yong_chong_dian_zhan",
    "ev_swap_cnt": "shenzhen_huan_dian_zhan",
    "ev_ebike_charge_cnt": "shenzhen_dian_dong_zi_dong_che_chong_dian_zhan",
}

ctx = pd.DataFrame({"h3_id": grid["h3_id"].values})

for col, stem in ev_layers.items():
    gdf = load_points(stem)
    ctx[col] = count_in_grid(gdf, col).values
    if gdf is not None:
        print(f"  {col}: {len(gdf):,} points, grids with >0: {(ctx[col] > 0).sum():,}")

# 其它能源站：仅保留名称/类型等字段中含充电、换电、新能源 的记录
OTHER = load_points("shenzhen_qi_ta_neng_yuan_zhan")
if OTHER is not None and len(OTHER):
    text_cols = [c for c in OTHER.columns if c != "geometry"]

    def row_text(r) -> str:
        return " ".join(str(r[c]) for c in text_cols if pd.notna(r[c]))

    kw = ("充电", "换电", "新能源")
    mask = OTHER.apply(lambda r: any(k in row_text(r) for k in kw), axis=1)
    ev_other = OTHER.loc[mask]
    print(f"  其它能源站: total {len(OTHER):,}, EV-keyword subset {len(ev_other):,}")
    ctx["ev_other_energy_cnt"] = count_in_grid(ev_other, "x").values
else:
    ctx["ev_other_energy_cnt"] = 0
    print("  其它能源站: 无或未筛选到电动车相关点")

KERB = load_points("shenzhen_shang_xia_ke_qu")
ctx["kerb_pickup_cnt"] = count_in_grid(KERB, "k").values
if KERB is not None:
    print(f"  kerb_pickup_cnt: {len(KERB):,} points")

ctx["ev_facility_total"] = (
    ctx["ev_charge_cnt"]
    + ctx["ev_dedicated_charge_cnt"]
    + ctx["ev_swap_cnt"]
    + ctx["ev_ebike_charge_cnt"]
    + ctx["ev_other_energy_cnt"]
)
print(f"\nGrids with any EV charging (incl. ebike): {(ctx['ev_facility_total'] > 0).sum():,}")

In [ ]:
# ============================================================
#  2) 通行形态：格网计数 + 80m buffer 环境带（质心是否落入）
# ============================================================

walk_stems = {
    "walk_street_cnt": "shenzhen_bu_xing_jie",
    "special_pass_cnt": "shenzhen_te_shu_tong_dao",
    "passage_fac_cnt": "shenzhen_tong_xing_she_shi",
}

walk_parts = []
for col, stem in walk_stems.items():
    gdf = load_points(stem)
    ctx[col] = count_in_grid(gdf, col).values
    if gdf is not None and len(gdf):
        walk_parts.append(gdf.to_crs(4326))
        print(f"  {col}: {len(gdf):,} points")

BUF_M = 80.0
if walk_parts:
    w_all = pd.concat(walk_parts, ignore_index=True)
    w_p = w_all.to_crs(PROJ_METRIC)
    buf_u = unary_union(w_p.geometry.buffer(BUF_M))
    buf_gdf = gpd.GeoDataFrame(geometry=[buf_u], crs=PROJ_METRIC).to_crs(4326)
    cent = grid.copy()
    cent["geometry"] = cent.geometry.centroid
    j = gpd.sjoin(cent[["h3_id", "geometry"]], buf_gdf, predicate="within", how="left")
    ctx["in_walk_env_buffer80m"] = j["index_right"].notna().astype(int).values
    print(f"  in_walk_env_buffer80m: {ctx['in_walk_env_buffer80m'].sum():,} grids")
else:
    ctx["in_walk_env_buffer80m"] = 0

ctx["walk_context_pts_total"] = ctx[list(walk_stems.keys())].sum(axis=1)

In [ ]:
# ============================================================
#  3) 公交/地铁：主数据 vs 11-公服 交叉校验（不重复写入格网距离）
# ============================================================

THRESH_M = 80.0
THRESH_DEG = THRESH_M / 111_000.0  # 近似，与 10 的平面 KDTree 一致量级


def transit_crosscheck(name_primary: str, gfs_stem: str):
    if not BUS_STOPS_PRIMARY.exists() or not METRO_STOPS_PRIMARY.exists():
        print("主数据公交/地铁缺失，跳过校验")
        return
    bus = gpd.read_file(BUS_STOPS_PRIMARY).to_crs(4326)
    metro = gpd.read_file(METRO_STOPS_PRIMARY).to_crs(4326)
    if name_primary == "bus":
        primary = bus
    else:
        primary = metro
    gfs = load_points(gfs_stem)
    if gfs is None or len(gfs) == 0:
        print(f"[{gfs_stem}] 无数据")
        return
    gfs = gfs.to_crs(4326)
    pc = np.column_stack([primary.geometry.x, primary.geometry.y])
    tree = cKDTree(pc)
    gc = np.column_stack([gfs.geometry.x, gfs.geometry.y])
    dist, _idx = tree.query(gc, distance_upper_bound=THRESH_DEG)
    matched = np.isfinite(dist) & (dist <= THRESH_DEG)
    n_m = int(matched.sum())
    pct = 100 * n_m / len(gfs)
    print(
        f"[{name_primary}] 主数据点数: {len(primary):,} | 11-公服 {gfs_stem}: {len(gfs):,} | "
        f"在 {THRESH_M:.0f}m 内找到主数据近邻: {n_m:,} ({pct:.1f}%)"
    )


transit_crosscheck("bus", "shenzhen_gong_jiao_che_zhan")
transit_crosscheck("metro", "shenzhen_di_tie_zhan")

rail = load_points("shenzhen_huo_che_zhan")
if rail is not None:
    print(f"[火车站] 11-公服点数: {len(rail):,}（仅作枢纽参考，不与 10 transit 树合并）")

# 格网仅保留「主数据」可达性时，请继续用 10 的 OD 结果；此处不写 dist_to_metro 第二套。


In [ ]:
# ============================================================
#  4) 桥隧立交 vs 05 障碍线：空间对照（不计入摩擦）
# ============================================================


def barrier_vs_gfs_points(gfs_stems, barrier_path: Path, label: str, max_m: float = 100.0):
    if not barrier_path.exists():
        print(f"{label}: barrier file missing")
        return
    lines = gpd.read_file(barrier_path).to_crs(PROJ_METRIC)
    pts = []
    for st in gfs_stems:
        g = load_points(st, quiet=True)
        if g is not None and len(g):
            pts.append(g.to_crs(PROJ_METRIC))
    if not pts:
        print(f"{label}: no 公服 points")
        return
    P = pd.concat(pts, ignore_index=True)
    j = gpd.sjoin_nearest(P, lines, how="left", max_distance=max_m, distance_col="d_m")
    hit = j["d_m"].notna()
    extra = (
        f"近邻距离中位数(m): {j.loc[hit, 'd_m'].median():.1f}"
        if hit.any()
        else "median n/a"
    )
    print(
        f"{label}: 公服点 {len(P):,} | OSM barrier 线 {len(lines):,} | "
        f"在 {max_m:.0f}m 内有近邻线: {hit.sum():,} ({100 * hit.mean():.1f}%) | {extra}"
    )


barrier_vs_gfs_points(["shenzhen_sui_dao"], BARRIER_TUNNELS, "隧道点 vs sz_barrier_tunnels")
barrier_vs_gfs_points(
    ["shenzhen_li_jiao_qiao", "shenzhen_qiao", "shenzhen_qiao_liang_sui_dao"],
    BARRIER_BRIDGES,
    "桥/立交/桥梁隧道点 vs sz_barrier_bridges",
)


In [ ]:
# ============================================================
#  5) 写出格网 GPKG
# ============================================================

out_gdf = grid.merge(ctx, on="h3_id", how="left")
out_path = OUT / "sz_waimai_context_grid.gpkg"
out_gdf.to_file(out_path, driver="GPKG")
print(f"\nWritten: {out_path}  ({len(out_gdf):,} rows)")
print("Columns:", [c for c in out_gdf.columns if c != "geometry"])